# Séance 8 — CNN : variations d'architecture

**Objectifs.**
- Retrouver la batch normalization (vue séance 5 en contexte MLP) en contexte convolutif
  (`nn.BatchNorm2d`).
- Comprendre et coder **from scratch** un bloc résiduel (ResNet) : pourquoi ça aide à
  entraîner des réseaux plus profonds.
- Comparer empiriquement un mini-VGG (séance 7) et une version à profondeur comparable munie
  de connexions résiduelles.

On repart du même dataset (Fashion-MNIST) et du même mini-VGG que la séance 7, pour isoler
l'effet de l'architecture des autres facteurs (données, taille du problème).


In [ ]:
# !pip install -q torch torchvision matplotlib

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, random_split

import torchvision
from torchvision import transforms

import matplotlib.pyplot as plt

from training_toolbox import Trainer, accuracy

torch.manual_seed(0)


## Partie 0 — Données (identique à la séance 7)

Code de chargement inchangé : on garde exactement le même dataset et le même découpage
train/val/test qu'à la séance précédente, pour ne comparer que les architectures.


In [ ]:
CLASSES = [
    "T-shirt/top", "Pantalon", "Pull", "Robe", "Manteau",
    "Sandale", "Chemise", "Basket", "Sac", "Bottine",
]

transform = transforms.ToTensor()

full_train = torchvision.datasets.FashionMNIST(
    root="./data", train=True, download=True, transform=transform
)
test_set = torchvision.datasets.FashionMNIST(
    root="./data", train=False, download=True, transform=transform
)

n_val = 5000
n_train = len(full_train) - n_val
train_set, val_set = random_split(full_train, [n_train, n_val])

train_loader = DataLoader(train_set, batch_size=128, shuffle=True)
val_loader = DataLoader(val_set, batch_size=256)
test_loader = DataLoader(test_set, batch_size=256)


## Partie 1 — Rappel : le mini-VGG de la séance 7

On repart de l'architecture codée (et entraînée) séance dernière — cette fois entièrement
fournie, ce n'est plus l'objet de cette séance.


In [ ]:
class MiniVGG(nn.Module):
    def __init__(self, n_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1), nn.ReLU(),
            nn.Conv2d(32, 32, kernel_size=3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.ReLU(),
            nn.Conv2d(64, 64, kernel_size=3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 7 * 7, 128), nn.ReLU(),
            nn.Linear(128, n_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))


## Partie 2 — Batch normalization en contexte convolutif

Rappel (séance 5) : la batch normalization normalise les activations (moyenne 0, variance 1
sur le batch), puis applique une transformation affine apprise. En contexte convolutif,
`nn.BatchNorm2d(C)` normalise **par canal** `C`, sur l'ensemble des positions spatiales et du
batch — un seul couple (moyenne, variance) par canal, pas par pixel.

**À vous de jouer.** Complétez `MiniVGGBatchNorm` ci-dessous : c'est le même mini-VGG que la
Partie 1, avec un `nn.BatchNorm2d` ajouté après chaque convolution et **avant** chaque `ReLU`
(ordre classique : conv → BN → activation).


In [ ]:
class MiniVGGBatchNorm(nn.Module):
    def __init__(self, n_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            # TODO : nn.BatchNorm2d(32), puis nn.ReLU()
            nn.Conv2d(32, 32, kernel_size=3, padding=1),
            # TODO : nn.BatchNorm2d(32), puis nn.ReLU()
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            # TODO : nn.BatchNorm2d(64), puis nn.ReLU()
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            # TODO : nn.BatchNorm2d(64), puis nn.ReLU()
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 7 * 7, 128), nn.ReLU(),
            nn.Linear(128, n_classes),
        )

    def forward(self, x):
        return self.classifier(self.features(x))


**Question.** `nn.BatchNorm2d` se comporte différemment en mode `train()` et en mode
`eval()` (tout comme `nn.Dropout`). Quelle statistique utilise-t-il dans chaque cas, et
pourquoi cette différence est-elle nécessaire ? (Indice : `model.train(train)` est déjà géré
pour vous dans `Trainer._run_epoch`.)


## Partie 3 — Un bloc résiduel, from scratch

**Le problème de dégradation.** Au-delà d'une certaine profondeur, empiler plus de couches
convolutives *dégrade* les performances — y compris sur le train set, ce qui n'est pas un
problème de sur-apprentissage mais un problème d'**optimisation** : le réseau a du mal à
apprendre, même la fonction identité, à travers de nombreuses couches non linéaires
empilées.

**L'idée du bloc résiduel (ResNet).** Plutôt que de faire apprendre à un bloc de couches une
fonction `H(x)` directement, on le force à apprendre le **résidu** `F(x) = H(x) - x`, via une
connexion "raccourci" (*skip connection*) qui ajoute l'entrée à la sortie :
`H(x) = F(x) + x`. Si la fonction identité est optimale, il suffit au bloc d'apprendre
`F(x) ≈ 0`, ce qui est beaucoup plus facile à atteindre par descente de gradient qu'apprendre
l'identité à travers des couches non linéaires.

**À coder : `ResidualBlock`.**

- **Chemin principal** :
  `Conv2d(in_c, out_c, 3, stride=stride, padding=1, bias=False)` → `BatchNorm2d(out_c)` →
  `ReLU` → `Conv2d(out_c, out_c, 3, stride=1, padding=1, bias=False)` → `BatchNorm2d(out_c)`
  (pas de ReLU à la toute fin du chemin principal : le ReLU final vient après l'addition).
- **Raccourci (shortcut)** : si `in_c != out_c` ou `stride != 1`, l'entrée `x` n'a pas la même
  forme que la sortie du chemin principal — on ne peut pas les additionner directement. On
  utilise alors une "projection" : `Conv2d(in_c, out_c, kernel_size=1, stride=stride,
  bias=False)` → `BatchNorm2d(out_c)`. Sinon, le raccourci est l'identité (`nn.Identity()`).
- **Sortie du bloc** : `ReLU(chemin_principal(x) + shortcut(x))`.

(`bias=False` sur les convolutions suivies d'une BatchNorm : le biais de la BN joue déjà ce
rôle, celui de la convolution serait redondant.)


In [ ]:
class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()

        # TODO 1 : chemin principal (2 convolutions 3x3 + BN, cf. description ci-dessus)
        self.conv1 = None  # <- Conv2d(in_channels, out_channels, 3, stride=stride, padding=1, bias=False)
        self.bn1 = None    # <- BatchNorm2d(out_channels)
        self.conv2 = None  # <- Conv2d(out_channels, out_channels, 3, stride=1, padding=1, bias=False)
        self.bn2 = None    # <- BatchNorm2d(out_channels)

        # Raccourci : déjà fourni, rien à faire ici.
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(out_channels),
            )
        else:
            self.shortcut = nn.Identity()

    def forward(self, x):
        # TODO 2 : chemin principal -- conv1 -> bn1 -> ReLU -> conv2 -> bn2 (SANS ReLU final ici)
        out = x  # <- à remplacer

        # TODO 3 : ajouter le raccourci, puis appliquer le ReLU final
        out = out  # <- à remplacer : F.relu(out + self.shortcut(x))
        return out


# Vérification rapide : un bloc qui change de résolution et de nombre de canaux
block = ResidualBlock(in_channels=32, out_channels=64, stride=2)
dummy = torch.randn(4, 32, 14, 14)
print("Entrée :", dummy.shape, "-> Sortie :", block(dummy).shape)  # attendu : (4, 64, 7, 7)


## Partie 4 — Un mini-ResNet, à profondeur comparable au mini-VGG

Le mini-VGG a 4 couches convolutives (2 blocs de 2 convolutions). On construit un réseau
avec le même nombre de convolutions, organisées en 2 blocs résiduels (chaque
`ResidualBlock` contient 2 convolutions) : la profondeur est comparable, seule
l'architecture change (skip connections + batch norm systématique).


In [ ]:
class MiniResNet(nn.Module):
    def __init__(self, n_classes=10):
        super().__init__()
        self.stem = nn.Conv2d(1, 32, kernel_size=3, padding=1, bias=False)
        self.block1 = ResidualBlock(32, 32, stride=1)
        self.pool1 = nn.MaxPool2d(2)
        self.block2 = ResidualBlock(32, 64, stride=1)
        self.pool2 = nn.MaxPool2d(2)
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 7 * 7, 128), nn.ReLU(),
            nn.Linear(128, n_classes),
        )

    def forward(self, x):
        x = self.stem(x)
        x = self.pool1(self.block1(x))
        x = self.pool2(self.block2(x))
        return self.classifier(x)


model_resnet = MiniResNet()
with torch.no_grad():
    out = model_resnet(torch.randn(2, 1, 28, 28))
print("Sortie MiniResNet :", out.shape)
print(f"Paramètres MiniResNet   : {sum(p.numel() for p in model_resnet.parameters()):,}")


## Partie 5 — Comparaison empirique

On entraîne les trois variantes (mini-VGG simple, mini-VGG + BatchNorm, MiniResNet) sur les
mêmes données, avec les mêmes hyperparamètres, et on compare les courbes d'apprentissage.


In [ ]:
def train_and_report(model, name, epochs=5):
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    loss_fn = nn.CrossEntropyLoss()
    trainer = Trainer(model, optimizer, loss_fn, metrics={"acc": accuracy})
    print(f"--- {name} ({sum(p.numel() for p in model.parameters()):,} paramètres) ---")
    history = trainer.fit(train_loader, val_loader, epochs=epochs, verbose=True)
    return history


histories = {}
histories["mini-VGG"] = train_and_report(MiniVGG(), "mini-VGG")
histories["mini-VGG + BatchNorm"] = train_and_report(MiniVGGBatchNorm(), "mini-VGG + BatchNorm")
histories["MiniResNet"] = train_and_report(MiniResNet(), "MiniResNet")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for name, h in histories.items():
    axes[0].plot(h["val_loss"], label=name)
    axes[1].plot(h["val_acc"], label=name)
axes[0].set_title("Validation loss"); axes[0].set_xlabel("epoch"); axes[0].legend()
axes[1].set_title("Validation accuracy"); axes[1].set_xlabel("epoch"); axes[1].legend()
plt.tight_layout()
plt.show()


**Questions de compréhension.**

- Classez les trois modèles par accuracy de validation, et par vitesse de convergence
  (nombre d'epochs pour atteindre une accuracy donnée). Les observations correspondent-elles
  à ce que vous attendiez ?
- Les architectures utilisées ici restent peu profondes (4 convolutions) : à cette
  profondeur, le "problème de dégradation" évoqué en cours (des réseaux *très* profonds, type
  ResNet à 50 ou 100+ couches, qui n'arrivent plus à apprendre l'identité) n'est pas forcément
  très marqué. Qu'observez-vous malgré tout comme différence de comportement entre le
  mini-VGG et le MiniResNet ?
- BatchNorm et skip connections sont deux mécanismes différents mais souvent combinés en
  pratique. Sur la base de vos courbes, essayez de séparer ce qui relève de l'un et de
  l'autre (indice : comparez d'abord mini-VGG vs mini-VGG+BN, puis mini-VGG+BN vs
  MiniResNet, qui contient BN *et* les skip connections).

## Pour aller plus loin (optionnel)

- Empiler 2 ou 3 `ResidualBlock` supplémentaires (profondeur plus réaliste) et observer si
  l'écart avec un mini-VGG "empilé" de la même façon (sans skip connections) se creuse.
- Essayer une architecture "pre-activation" (BN → ReLU → Conv, plutôt que Conv → BN → ReLU) :
  c'est la variante proposée dans les versions ultérieures de ResNet.
- *(Non traité dans ce TP, cf. `ConvNets_contd_nocorr.ipynb` du cours précédent si vous
  voulez creuser)* : visualisation des filtres appris en profondeur, et exemples
  adversariaux (FGSM) — intéressant mais indépendant du sujet "architecture" de cette
  séance, laissé de côté ici faute de temps.
